# Water Quality Prediction: Copy Benchmark Model

This notebook mirrors the Benchmark_Model_Notebook workflow, but uses the combined EC/TA/DRP training datasets from `Datasets_Ours/Final Datasets`.

In [ ]:
# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

# Geospatial raster data handling with CRS support
import rioxarray as rxr

# Raster operations and spatial windowing
import rasterio
from rasterio.windows import Window

# Feature preprocessing and data splitting
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.spatial import cKDTree

# Machine Learning
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
from pystac.extensions.eo import EOExtension as eo

from datetime import date
from tqdm import tqdm
import os

In [ ]:
# Resolve project root robustly
cwd = os.getcwd()
cwd_base = os.path.basename(cwd)
if cwd_base == "models":
    PROJECT_ROOT = os.path.abspath(os.path.join(cwd, "..", ".."))
elif cwd_base == "Notebooks_Ours":
    PROJECT_ROOT = os.path.abspath(os.path.join(cwd, ".."))
else:
    PROJECT_ROOT = os.path.abspath(cwd)

DATA_DIR = os.path.join(PROJECT_ROOT, "Datasets_Ours", "Final Datasets")

# Load training datasets (DRP, EC, TA)
drp_train = pd.read_csv(os.path.join(DATA_DIR, "drp_training_complete.csv"))
ec_train = pd.read_csv(os.path.join(DATA_DIR, "ec_training_complete.csv"))
ta_train = pd.read_csv(os.path.join(DATA_DIR, "ta_training_complete.csv"))

# Standardize column names to match benchmark expectations
rename_map = {
    "latitude": "Latitude",
    "longitude": "Longitude",
    "sample_date": "Sample Date",
    "total_alkalinity": "Total Alkalinity",
    "electrical_conductance": "Electrical Conductance",
    "dissolved_reactive_phosphorus": "Dissolved Reactive Phosphorus",
}

drp_train = drp_train.rename(columns={k: v for k, v in rename_map.items() if k in drp_train.columns})
ec_train = ec_train.rename(columns={k: v for k, v in rename_map.items() if k in ec_train.columns})
ta_train = ta_train.rename(columns={k: v for k, v in rename_map.items() if k in ta_train.columns})

# Merge targets onto a single training table (join on lat/lon and sample_date if present)
join_keys = ["Latitude", "Longitude"]
if "Sample Date" in drp_train.columns and "Sample Date" in ec_train.columns and "Sample Date" in ta_train.columns:
    join_keys = ["Latitude", "Longitude", "Sample Date"]

wq_data = drp_train.copy()

# Ensure EC target exists
if "Electrical Conductance" not in wq_data.columns and "Electrical Conductance" in ec_train.columns:
    wq_data = wq_data.merge(ec_train[join_keys + ["Electrical Conductance"]], on=join_keys, how="inner")

# Ensure TA target exists
if "Total Alkalinity" not in wq_data.columns and "Total Alkalinity" in ta_train.columns:
    wq_data = wq_data.merge(ta_train[join_keys + ["Total Alkalinity"]], on=join_keys, how="inner")

# Ensure DRP target exists
if "Dissolved Reactive Phosphorus" not in wq_data.columns and "Dissolved Reactive Phosphorus" in drp_train.columns:
    wq_data = wq_data.merge(drp_train[join_keys + ["Dissolved Reactive Phosphorus"]], on=join_keys, how="inner")

wq_data.head()

In [ ]:
# Handle missing values (same as benchmark)
wq_data = wq_data.fillna(wq_data.median(numeric_only=True))

# Retain only the benchmark feature set
wq_data = wq_data[['swir22','NDMI','MNDWI','pet', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

wq_data.isna().sum()

In [ ]:
def split_data(X, y, test_size=0.3, random_state=42):
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

def scale_data(X_train, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler

def train_model(X_train_scaled, y_train):
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train_scaled, y_train)
    return model

def evaluate_model(model, X_scaled, y_true, dataset_name="Test"):
    y_pred = model.predict(X_scaled)
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"\n{dataset_name} Evaluation:")
    print(f"R²: {r2:.3f}")
    print(f"RMSE: {rmse:.3f}")
    return y_pred, r2, rmse


def run_pipeline(X, y, param_name="Parameter"):
    print(f"\n{'='*60}")
    print(f"Training Model for {param_name}")
    print(f"{'='*60}")
    
    # Split data
    X_train, X_test, y_train, y_test = split_data(X, y)
    
    # Scale
    X_train_scaled, X_test_scaled, scaler = scale_data(X_train, X_test)
    
    # Train
    model = train_model(X_train_scaled, y_train)
    
    # Evaluate (in-sample)
    y_train_pred, r2_train, rmse_train = evaluate_model(model, X_train_scaled, y_train, "Train")
    
    # Evaluate (out-sample)
    y_test_pred, r2_test, rmse_test = evaluate_model(model, X_test_scaled, y_test, "Test")
    
    # Return summary
    results = {
        "Parameter": param_name,
        "R2_Train": r2_train,
        "RMSE_Train": rmse_train,
        "R2_Test": r2_test,
        "RMSE_Test": rmse_test
    }
    return model, scaler, pd.DataFrame([results])

In [ ]:
X = wq_data.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])

y_TA = wq_data['Total Alkalinity']
y_EC = wq_data['Electrical Conductance']
y_DRP = wq_data['Dissolved Reactive Phosphorus']

model_TA, scaler_TA, results_TA = run_pipeline(X, y_TA, "Total Alkalinity")
model_EC, scaler_EC, results_EC = run_pipeline(X, y_EC, "Electrical Conductance")
model_DRP, scaler_DRP, results_DRP = run_pipeline(X, y_DRP, "Dissolved Reactive Phosphorus")

In [ ]:
results_summary = pd.concat([results_TA, results_EC, results_DRP], ignore_index=True)
results_summary

In [ ]:
# Submission
# Reading the coordinates for the submission
submission_template = pd.read_csv(os.path.join(PROJECT_ROOT, "submission_template.csv"))

# Validation feature dataset (use EC validation as feature source)
ec_val = pd.read_csv(os.path.join(DATA_DIR, "ec_validation.csv"))

# Standardize column names to match benchmark expectations
if "latitude" in ec_val.columns:
    ec_val = ec_val.rename(columns={"latitude": "Latitude", "longitude": "Longitude"})

# Extract features for submission (same as benchmark)
submission_val_data = ec_val.loc[:, ['swir22','NDMI','MNDWI','pet']]

# Predict for Total Alkalinity
X_sub_scaled_TA = scaler_TA.transform(submission_val_data)
pred_TA_submission = model_TA.predict(X_sub_scaled_TA)

# Predict for Electrical Conductance
X_sub_scaled_EC = scaler_EC.transform(submission_val_data)
pred_EC_submission = model_EC.predict(X_sub_scaled_EC)

# Predict for Dissolved Reactive Phosphorus
X_sub_scaled_DRP = scaler_DRP.transform(submission_val_data)
pred_DRP_submission = model_DRP.predict(X_sub_scaled_DRP)

submission_df = pd.DataFrame({
    'Longitude': submission_template['Longitude'].values,
    'Latitude': submission_template['Latitude'].values,
    'Sample Date': submission_template['Sample Date'].values,
    'Total Alkalinity': pred_TA_submission,
    'Electrical Conductance': pred_EC_submission,
    'Dissolved Reactive Phosphorus': pred_DRP_submission
})

# Save submission
out_path = os.path.join(PROJECT_ROOT, "Submissions", "copy_submission.csv")
submission_df.to_csv(out_path, index=False)

submission_df.head()